# Control test — does the model read ANY line chart with ANY values and ANY axes?

Six new DxR line charts filled with **random numbers that exist nowhere else** (`ground_truth/control/`, fixed and
hashed in `MANIFEST.json` before recording). None of them, nor their values, was ever used in training.

| Recording | Axes | Values |
|---|---|---|
| `line_control_A` | Year × life expectancy, 4 countries (same layout as Gapminder) | 48 |
| `line_control_B` | Year × life expectancy, 3 countries, every 10 years | 18 |
| `line_control_C` | Year × life expectancy, 6 countries (harder) | 78 |
| `linexy_control_D` | **Month × sales (thousand units)**, 3 products | 36 |
| `linexy_control_E` | **Hour × energy use (kWh)**, 4 stations | 48 |
| `linexy_control_F` | **Day × price ($)**, 3 items, decimals | 36 |

`line_*` charts are read with the life-expectancy prompt (header Country,Year,Life_expectancy); `linexy_*` charts with
a general prompt (header Series,X,Y: legend name, x value, y value).

| Cell | What it does | Time (A100 · T4) |
|---|---|---|
| 1–4 | install, Drive, code + truth tables, settings | ~7 min |
| 5 | upload the recordings (kept in Drive) | 2 min |
| 6 | **run-4 model** on every recording | ~2 · 5 min per video |
| 7 | untrained Qwen3.5-9B on every recording | same |
| 8 | results table + proof video and slide image per recording | ~2 min |
| 9 | **run 5** (only if the other-axes charts need it): train on any-axes + life-expectancy synthetic line charts | ~70 min · ~3.5 h |
| 10 | run-5 model on every recording, then run cell 8 again | ~2 · 5 min per video |

In [ ]:
# 1 · Install
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install -q --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install -q transformers==5.2.0
!unsloth install-kernels
!uv pip install -q --no-deps --upgrade "torchao>=0.16.0"
!python3 -c "import unsloth, transformers; print('INSTALL OK - unsloth ready, transformers', transformers.__version__)"

In [ ]:
# 2 · Google Drive + project zip (the one run 4 saved in Drive; asked only if it is not there)
from google.colab import files, drive
import zipfile, os, glob, shutil, json, csv
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/dxr_run4"
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists("/content/dxr/vlm_eval/prompts.md"):
    src = f"{DRIVE}/dxr_video_upload.zip"
    if not os.path.exists(src):
        print("Choose dxr_video_upload.zip from your Desktop")
        up = files.upload(); shutil.copy(next(iter(up)), src)
    zipfile.ZipFile(src).extractall("/content/dxr")
os.chdir("/content/dxr")
!ls
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3 · Updated code + control truth tables (fixed before recording; see MANIFEST.json)
%cd /content/dxr
import base64, io, tarfile
CODE = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode="r:xz").extractall("/content/dxr")
!python3 vlm_eval/run_local_vlm.py
print(open("ground_truth/control/MANIFEST.json").read())

In [ ]:
# 4 · Settings
%cd /content/dxr
import torch
ADAPTER = f"{DRIVE}/lora_qwen35_line_r4"            # run-4 model saved by the Run 4 notebook
if not os.path.exists(f"{ADAPTER}/adapter_config.json"):
    ADAPTER = "/content/drive/MyDrive/dxr_lora/lora_qwen35_line"     # fall back to run 3
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "No trained adapter found in Drive (dxr_run4/lora_qwen35_line_r4)"
NAME = "qwen35_r4_line" if ADAPTER.endswith("_r4") else "qwen35_r3_line"
ADAPTER5, NAME5 = f"{DRIVE}/lora_qwen35_anyline_r5", "qwen35_r5_anyline"   # made by cell 9
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
SIDE, X = (800, "") if gb > 30 else (640, "--load-4bit")
print(f"model {ADAPTER} | GPU {gb:.0f} GB | {SIDE} px {X or '16-bit'}")

In [ ]:
# 5 · Your recordings: line_control_A/B/C.mp4 and linexy_control_D/E/F.mp4 (the name links each video to its truth table)
%cd /content/dxr
VDIR = f"{DRIVE}/control_videos"
os.makedirs(VDIR, exist_ok=True)
have = sorted(os.path.basename(p) for p in glob.glob(f"{VDIR}/*.*"))
print("already in Drive:", have or "none")
print("Choose any new recordings (or Cancel)")
try:
    for n in files.upload(): shutil.copy(n, f"{VDIR}/{n}")
except Exception as e:
    print("no upload:", e)
!python3 finetune_video/sample_real_videos.py {VDIR} --charts line linexy --width {SIDE} --out real_clips_ctrl
VIDEOS = [json.loads(l)["video"] for l in open("real_clips_ctrl/real.jsonl")]
print("testing:", VIDEOS)
print("no truth table (renamed?):", [v for v in VIDEOS if v != "line_clear" and not os.path.exists(f"ground_truth/control/{v}.csv")] or "none")

In [ ]:
# 6 · Run-4 model on every recording (one answer per video, same as the 100 % result)
%cd /content/dxr
!python finetune_video/eval_video.py --model {ADAPTER} --name {NAME} --charts line linexy --real real_clips_ctrl/real.jsonl --videos {" ".join(VIDEOS)} --val-n 0 --max-side {SIDE} {X} --results results_control.csv
shutil.copy("results_control.csv", DRIVE); shutil.copytree("predictions", f"{DRIVE}/predictions_control", dirs_exist_ok=True)

In [ ]:
# 7 · Untrained Qwen3.5-9B on the same recordings (what fine-tuning adds)
%cd /content/dxr
!python finetune_video/eval_video.py --model unsloth/Qwen3.5-9B --name qwen35_base --charts line linexy --real real_clips_ctrl/real.jsonl --videos {" ".join(VIDEOS)} --val-n 0 --max-side {SIDE} {X} --results results_control.csv
shutil.copy("results_control.csv", DRIVE); shutil.copytree("predictions", f"{DRIVE}/predictions_control", dirs_exist_ok=True)

In [ ]:
# 8 · Results table + proof video and slide image per recording (proof uses the newest fine-tuned model)
%cd /content/dxr
from IPython.display import Markdown, Image as IPImage, display
import sys; sys.path.insert(0, "control_test"); import make_proof_video as mpv
rows = [r for r in csv.DictReader(open("results_control.csv")) if r["set"] == "real"]
models = [m for m in ["qwen35_base", NAME, NAME5] if any(r["model"] == m for r in rows)]
BEST = models[-1]
label = {"qwen35_base": "Untrained Qwen3.5-9B", NAME: f"Run 4 ({NAME})", NAME5: f"Run 5 ({NAME5})"}
truth_of = lambda v: f"ground_truth/control/{v}.csv" if os.path.exists(f"ground_truth/control/{v}.csv") else "ground_truth/line_gapminder_lifeexp.csv"
os.makedirs("proof", exist_ok=True)
lines = ["| Recording | Axes | True values | " + " | ".join(label[m] for m in models) + f" | Within 2 % ({BEST}) | Mean error | Missing |",
         "|---|---|---|" + "---|" * (len(models) + 3)]
man = json.load(open("ground_truth/control/MANIFEST.json"))["charts"]
for v in VIDEOS:
    get = lambda m: next((r for r in rows if r["model"] == m and r["clip"] == v), None)
    pct = lambda r: f"{100 * float(r['value_acc']):.1f} %" if r else "–"
    pred = f"predictions/video_{BEST}/real_video_{v}.csv"
    stt = mpv.stats(mpv.compare(mpv.read(truth_of(v)), mpv.read(pred))[1]) if os.path.exists(pred) else None
    extra = f"{stt['w2']} / {stt['n']} | {stt['mre']:.2f} % | {stt['missing']}" if stt and stt["mre"] is not None else "– | – | –"
    axes = f"{man[v]['x_axis']} × {man[v]['y_axis']}" if v in man else "Year × Life expectancy"
    lines.append(f"| {v} | {axes} | {stt['n'] if stt else '–'} | " + " | ".join(("**" + pct(get(m)) + "**") if m == BEST else pct(get(m)) for m in models) + f" | {extra} |")
    vid = next(iter(glob.glob(f"{VDIR}/{v}.*")))
    if os.path.exists(pred):
        !python3 control_test/make_proof_video.py --video "{vid}" --truth {truth_of(v)} --pred {pred} --frames real_clips_ctrl/{v} --model "{label[BEST]}: Qwen3.5-9B + LoRA, trained on synthetic charts only" --out proof/{v}_proof.mp4
table = "\n".join(lines)
display(Markdown("### Share of true values read within 5 %\n\n" + table))
for v in VIDEOS:
    if os.path.exists(f"proof/{v}_proof_result.png"): display(IPImage(f"proof/{v}_proof_result.png", width=900))
open("proof/RESULTS_control.md", "w").write("### Share of true values read within 5 %\n\n" + table + "\n")
shutil.copytree("proof", f"{DRIVE}/proof", dirs_exist_ok=True)
!zip -qr control_results.zip proof results_control.csv predictions ground_truth/control
shutil.copy("control_results.zip", DRIVE)
files.download("control_results.zip")
print("Also in Google Drive: MyDrive/dxr_run4/proof/ and control_results.zip")

In [ ]:
# 9 · Run 5 (optional): one model for ANY line chart. Synthetic only: 50 % any-axes charts (Series,X,Y), 50 % life-expectancy charts
%cd /content/dxr
TRAIN5, STEPS5, MAXLEN5 = (2000, 250, 12288) if not X else (1500, 188, 6144)
if not os.path.exists(f"{ADAPTER5}/adapter_config.json"):
    !python finetune_video/make_video_dataset.py --strict --charts line linexy --train {TRAIN5} --val 30 --size {SIDE}x{SIDE*3//4} --out finetune_video/data_anyline
    !python finetune_video/train_video_lora.py --base unsloth/Qwen3.5-9B --data finetune_video/data_anyline --steps {STEPS5} --max-side {SIDE} --max-length {MAXLEN5} --out lora_qwen35_anyline_r5 {X}
    shutil.copytree("lora_qwen35_anyline_r5", ADAPTER5, dirs_exist_ok=True)
print(open(f"{ADAPTER5}/training_info.json").read())

In [ ]:
# 10 · Run-5 model on every recording; then run cell 8 again (it switches the proof videos to run 5)
%cd /content/dxr
!python finetune_video/eval_video.py --model {ADAPTER5} --name {NAME5} --charts line linexy --real real_clips_ctrl/real.jsonl --videos {" ".join(VIDEOS)} --val-n 0 --max-side {SIDE} {X} --results results_control.csv
shutil.copy("results_control.csv", DRIVE); shutil.copytree("predictions", f"{DRIVE}/predictions_control", dirs_exist_ok=True)

**Order:** cells 1–8 first. If the run-4 model is weak on the other-axes charts (D–F), run cells 9, 10 and 8 again.

**What goes in the presentation:** the proof videos `proof/<recording>_proof.mp4` (title card with the truth table's
hash, the raw unedited Quest recording, the 8 frames the model saw, the result) and the slide images
`proof/<recording>_proof_result.png`. The original raw recordings are in Drive `dxr_run4/control_videos/`.
The model's raw text answers are in `predictions/video_<name>/real_video_<recording>.json`.